In [1]:
# Cell 1: Imports and Configurations
import os
import time
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LSTM, Conv1D, MaxPooling1D, Flatten, Input
from tensorflow.keras.optimizers import Adam
import warnings

warnings.filterwarnings("ignore")

# Set random seed for reproducibility
np.random.seed(42)
import tensorflow as tf

tf.random.set_seed(42)

# Define paths
DATA_PATH = r"D:\study\Uni_Matrial\Final_Project\Digital_Twin\01_AI_and_Data\data\raw\CMAPSSData"
DATASETS = ["FD001", "FD002", "FD003"]
WINDOW_SIZE = 30

c:\Users\kira\miniconda3\envs\digital_twin\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# Cell 2: Data Preprocessing (Aligned with your previous sensor selection)
def load_and_preprocess_data(dataset_name, data_path, window_size):
    # Column names based on NASA CMAPSS documentation
    index_names = ["unit_nr", "time_cycles"]
    setting_names = ["setting_1", "setting_2", "setting_3"]
    sensor_names = ["s_{}".format(i) for i in range(1, 22)]
    col_names = index_names + setting_names + sensor_names

    # Load data
    train_df = pd.read_csv(
        os.path.join(data_path, f"train_{dataset_name}.txt"),
        sep="\s+",
        header=None,
        names=col_names,
    )
    test_df = pd.read_csv(
        os.path.join(data_path, f"test_{dataset_name}.txt"),
        sep="\s+",
        header=None,
        names=col_names,
    )
    # RUL ground truth for test sets
    rul_df = pd.read_csv(
        os.path.join(data_path, f"RUL_{dataset_name}.txt"),
        sep="\s+",
        header=None,
        names=["RUL"],
    )

    # --- Step 1: Calculate and Clip RUL for training data ---
    max_cycle = train_df.groupby("unit_nr")["time_cycles"].max().reset_index()
    max_cycle.columns = ["unit_nr", "max_life"]
    train_df = train_df.merge(max_cycle, on="unit_nr", how="left")
    train_df["RUL"] = train_df["max_life"] - train_df["time_cycles"]

    # Applying the 125 clip as per your standard technique
    train_df["RUL"] = train_df["RUL"].clip(upper=125)
    train_df.drop("max_life", axis=1, inplace=True)

    # --- Step 2: Feature Selection (Your specific choice) ---
    # Dropping: setting_3, s_1, s_5, s_6, s_10, s_16, s_18, s_19
    # This leaves 16 features (2 settings + 14 sensors)
    cols_to_drop = ["setting_3", "s_1", "s_5", "s_6", "s_10", "s_16", "s_18", "s_19"]

    # Final features to be used in training/testing
    features = [c for c in (setting_names + sensor_names) if c not in cols_to_drop]

    # Scale features
    scaler = MinMaxScaler()
    train_df[features] = scaler.fit_transform(train_df[features])
    test_df[features] = scaler.transform(test_df[features])

    # --- Step 3: Sequence Generation (Sliding Window) ---
    def gen_sequence(id_df, seq_length, seq_cols):
        data_matrix = id_df[seq_cols].values
        num_elements = data_matrix.shape[0]
        for start, stop in zip(
            range(0, num_elements - seq_length), range(seq_length, num_elements)
        ):
            yield data_matrix[start:stop, :]

    def gen_labels(id_df, seq_length, label):
        data_matrix = id_df[label].values
        num_elements = data_matrix.shape[0]
        return data_matrix[seq_length:num_elements, :]

    # Generate training sequences
    X_train = []
    y_train = []
    for unit_id in train_df["unit_nr"].unique():
        # Get sequences for each engine unit
        unit_data = train_df[train_df["unit_nr"] == unit_id]
        X_train.extend(list(gen_sequence(unit_data, window_size, features)))
        y_train.extend(list(gen_labels(unit_data, window_size, ["RUL"])))

    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)

    # Generate testing sequences (taking the last window for RUL prediction)
    X_test = []
    for unit_id in test_df["unit_nr"].unique():
        unit_data = test_df[test_df["unit_nr"] == unit_id][features].values
        if len(unit_data) >= window_size:
            X_test.append(unit_data[-window_size:, :])
        else:
            # Padding if engine data is shorter than window size
            pad_shape = (window_size - len(unit_data), len(features))
            padded_data = np.vstack([np.zeros(pad_shape), unit_data])
            X_test.append(padded_data)

    X_test = np.asarray(X_test)
    y_test = rul_df["RUL"].values

    return X_train, y_train, X_test, y_test, len(features)

In [3]:
# Cell 3: Model Definitions
def build_rf_model():
    return RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)


def build_cnn_model(window_size, num_features):
    model = Sequential(
        [
            Conv1D(
                filters=64,
                kernel_size=3,
                activation="relu",
                input_shape=(window_size, num_features),
            ),
            MaxPooling1D(pool_size=2),
            Flatten(),
            Dense(50, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    return model


def build_standard_lstm(window_size, num_features):
    model = Sequential(
        [
            LSTM(64, return_sequences=True, input_shape=(window_size, num_features)),
            LSTM(32, return_sequences=False),
            Dense(32, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    return model


def build_proposed_digital_twin(base_model, window_size, num_features):
    # Simulate Transfer Learning/Quick Calibration by freezing base layers
    # In a real scenario, this loads pre-trained FD001 weights and fine-tunes on a small subset
    new_model = Sequential()
    for layer in base_model.layers[:-2]:  # Keep LSTM layers
        layer.trainable = False
        new_model.add(layer)
    new_model.add(Dense(32, activation="relu"))
    new_model.add(Dense(1))
    new_model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    return new_model

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, LSTM, Conv1D, Flatten, Input, MaxPooling1D
from tensorflow.keras.optimizers import Adam

# Configuration and Hyperparameters
SEQUENCE_LENGTH = 30
BATCH_SIZE = 64
EPOCHS_FINETUNE = 15
EPOCHS_BASELINE = 20
DATA_DIR = r"D:\study\Uni_Matrial\Final_Project\Digital_Twin\01_AI_and_Data\data\raw\CMAPSSData"
BASE_MODEL_PATH = r"D:\study\Uni_Matrial\Final_Project\Digital_Twin\01_AI_and_Data\saved_models\calibrated_model.keras"

# Columns for C-MAPSS dataset
INDEX_COLUMNS = ["unit_number", "time_in_cycles"]
SETTING_COLUMNS = ["setting_1", "setting_2", "setting_3"]
SENSOR_COLUMNS = ["sensor_{}".format(i) for i in range(1, 22)]
ALL_COLUMNS = INDEX_COLUMNS + SETTING_COLUMNS + SENSOR_COLUMNS


# 1. Data Processing Functions
def add_remaining_useful_life(df, clip_rul=125):
    # Calculate RUL based on max cycles per unit
    rul = pd.DataFrame(df.groupby("unit_number")["time_in_cycles"].max()).reset_index()
    rul.columns = ["unit_number", "max_cycles"]
    df = df.merge(rul, on=["unit_number"], how="left")
    df["RUL"] = df["max_cycles"] - df["time_in_cycles"]
    df.drop("max_cycles", axis=1, inplace=True)
    # Piece-wise linear RUL target
    df["RUL"] = df["RUL"].clip(upper=clip_rul)
    return df


def process_data(dataset_name):
    # Construct file paths
    train_file = os.path.join(DATA_DIR, f"train_{dataset_name}.txt")
    test_file = os.path.join(DATA_DIR, f"test_{dataset_name}.txt")
    rul_file = os.path.join(DATA_DIR, f"RUL_{dataset_name}.txt")

    # Load data
    train_df = pd.read_csv(train_file, sep="\s+", header=None, names=ALL_COLUMNS)
    test_df = pd.read_csv(test_file, sep="\s+", header=None, names=ALL_COLUMNS)
    truth_df = pd.read_csv(rul_file, sep="\s+", header=None, names=["RUL"])

    # Add RUL to training data
    train_df = add_remaining_useful_life(train_df)

    # Normalize data (fit on train, transform on train and test)
    scaler = MinMaxScaler()
    train_df[SENSOR_COLUMNS] = scaler.fit_transform(train_df[SENSOR_COLUMNS])
    test_df[SENSOR_COLUMNS] = scaler.transform(test_df[SENSOR_COLUMNS])

    return train_df, test_df, truth_df


def generate_sequences(df, sequence_length, feature_cols):
    # Generate overlapping windows for time-series models
    X, y = [], []
    for unit in df["unit_number"].unique():
        unit_data = df[df["unit_number"] == unit]
        data_matrix = unit_data[feature_cols].values
        rul_array = unit_data["RUL"].values

        for i in range(len(unit_data) - sequence_length + 1):
            X.append(data_matrix[i : i + sequence_length])
            y.append(rul_array[i + sequence_length - 1])

    return np.array(X), np.array(y)


def generate_test_sequences(test_df, truth_df, sequence_length, feature_cols):
    # Generate exactly one sequence per unit for the final test evaluation
    X, y = [], []
    for i, unit in enumerate(test_df["unit_number"].unique()):
        unit_data = test_df[test_df["unit_number"] == unit]
        if len(unit_data) >= sequence_length:
            X.append(unit_data[feature_cols].values[-sequence_length:])
            y.append(truth_df.iloc[i].values[0])
    return np.array(X), np.array(y)


# 2. Model Definitions
def build_random_forest():
    return RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)


def build_standard_cnn(input_shape):
    model = Sequential(
        [
            Input(shape=input_shape),
            Conv1D(filters=64, kernel_size=3, activation="relu"),
            MaxPooling1D(pool_size=2),
            Conv1D(filters=128, kernel_size=3, activation="relu"),
            Flatten(),
            Dense(64, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer="adam", loss="mse")
    return model


def build_standard_lstm(input_shape):
    model = Sequential(
        [
            Input(shape=input_shape),
            LSTM(64, return_sequences=True),
            LSTM(32, return_sequences=False),
            Dense(32, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer="adam", loss="mse")
    return model


def finetune_digital_twin(base_model_path, X_train, y_train, input_shape):
    # Load the base model trained on FD004
    # Hard failure if the model file is missing — no silent fallback to untrained weights
    if not os.path.exists(base_model_path):
        raise FileNotFoundError(
            f"Base model not found at: {base_model_path}\n"
            "Please train and save the universal model before running fine-tuning."
        )
    model = load_model(base_model_path)

    # Freeze the early layers to retain FD004 knowledge, fine-tune the dense layers
    for layer in model.layers[:-2]:
        layer.trainable = False

    # Recompile with a lower learning rate for fine-tuning
    model.compile(optimizer=Adam(learning_rate=0.0001), loss="mse")

    # Train on the new specific dataset (FD001, FD002, or FD003)
    model.fit(
        X_train,
        y_train,
        epochs=EPOCHS_FINETUNE,
        batch_size=BATCH_SIZE,
        verbose=0,
        validation_split=0.1,
    )
    return model


# 3. Evaluation Function
def evaluate_model(model, X_test, y_test, is_rf=False):
    if is_rf:
        # Flatten sequences for Random Forest (it doesn't take 3D inputs natively)
        X_test_flat = X_test.reshape(X_test.shape[0], -1)
        predictions = model.predict(X_test_flat)
    else:
        predictions = model.predict(X_test, verbose=0).flatten()

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    return rmse


# 4. Main Execution Pipeline
def main():
    print("--- Starting RUL Evaluation Pipeline ---")

    input_shape = (SEQUENCE_LENGTH, len(SENSOR_COLUMNS))
    base_model_path = r"D:\study\Uni_Matrial\Final_Project\Digital_Twin\01_AI_and_Data\saved_models\calibrated_model.keras"
    results_summary = {}

    # Step 1: Evaluate Standard LSTM strictly on test_FD004.txt to check true performance
    print("\n[Step 1] Evaluating Standard LSTM on FD004 Test Data...")
    try:
        train_df, test_df, truth_df = process_data("FD004")
        X_test_4, y_test_4 = generate_test_sequences(
            test_df, truth_df, SEQUENCE_LENGTH, SENSOR_COLUMNS
        )

        # Simulating standard LSTM model loaded/trained
        lstm_model = build_standard_lstm(input_shape)
        # lstm_model.load_weights('standard_lstm_weights.h5') # Load actual weights here

        rmse_lstm_4 = evaluate_model(lstm_model, X_test_4, y_test_4)
        print(f"Standard LSTM (No Calibration) RMSE on test_FD004: {rmse_lstm_4:.2f}")
    except Exception as e:
        print(f"Skipping FD004 LSTM test due to missing data: {e}")

    # Step 2: Loop through FD001, FD002, FD003 for Baselines and Fine-Tuning
    datasets = ["FD001", "FD002", "FD003"]

    for dataset in datasets:
        print(f"\n--- Processing {dataset} ---")
        try:
            train_df, test_df, truth_df = process_data(dataset)

            # Prepare sequences
            X_train, y_train = generate_sequences(
                train_df, SEQUENCE_LENGTH, SENSOR_COLUMNS
            )
            X_test, y_test = generate_test_sequences(
                test_df, truth_df, SEQUENCE_LENGTH, SENSOR_COLUMNS
            )

            dataset_results = {}

            # Baseline 1: Random Forest
            print(f"Training Random Forest (Baseline) on {dataset}...")
            rf_model = build_random_forest()
            X_train_flat = X_train.reshape(X_train.shape[0], -1)
            rf_model.fit(X_train_flat, y_train)
            dataset_results["Random Forest"] = evaluate_model(
                rf_model, X_test, y_test, is_rf=True
            )

            # Baseline 2: Standard CNN
            print(f"Training Standard CNN (Baseline) on {dataset}...")
            cnn_model = build_standard_cnn(input_shape)
            cnn_model.fit(
                X_train,
                y_train,
                epochs=EPOCHS_BASELINE,
                batch_size=BATCH_SIZE,
                verbose=0,
            )
            dataset_results["Standard CNN"] = evaluate_model(cnn_model, X_test, y_test)

            # Proposed Edge-Digital Twin (Fine-tuning)
            print(f"Fine-tuning Proposed Edge-Digital Twin on {dataset}...")
            dt_model = finetune_digital_twin(
                base_model_path, X_train, y_train, input_shape
            )
            dataset_results["Proposed Edge-Digital Twin"] = evaluate_model(
                dt_model, X_test, y_test
            )

            results_summary[dataset] = dataset_results

            # Print intermediate results
            print(f"Results for {dataset}:")
            for model_name, rmse in dataset_results.items():
                print(f"  - {model_name}: RMSE = {rmse:.2f}")

        except Exception as e:
            print(f"Could not process {dataset}. Error: {e}")

    # Step 3: Final Output Comparison
    print("\n=== Final Results Summary (RMSE) ===")
    for dataset, results in results_summary.items():
        print(f"\n{dataset}:")
        for model_name, rmse in results.items():
            print(f"  {model_name}: {rmse:.2f}")


if __name__ == "__main__":
    main()

--- Starting RUL Evaluation Pipeline ---

[Step 1] Evaluating Standard LSTM on FD004 Test Data...
Standard LSTM (No Calibration) RMSE on test_FD004: 98.79

--- Processing FD001 ---
Training Random Forest (Baseline) on FD001...


KeyboardInterrupt: 